In [44]:
print("OK")

OK


In [45]:
import pandas as pd
from evidently.dashboard import Dashboard
from evidently.tabs import DataDriftTab, CatTargetDriftTab
from evidently.model_profile import Profile
from evidently.profile_sections import DataDriftProfileSection
from evidently.pipeline.column_mapping import ColumnMapping

## Load out Boston data

In [33]:
usvisa = pd.read_csv("Visadataset.csv")

In [34]:
usvisa .head()

,case_id,continent,education_of_employee,has_job_experience,requires_job_training,no_of_employees,yr_of_estab,region_of_employment,prevailing_wage,unit_of_wage,full_time_position,case_status
0,EZYV01,Asia,High School,N,N,14513,2007,West,592.2029,Hour,Y,Denied
1,EZYV02,Asia,Master's,Y,N,2412,2002,Northeast,83425.6500,Year,Y,Certified
2,EZYV03,Asia,Bachelor's,N,Y,44444,2008,West,122996.8600,Year,Y,Denied
3,EZYV04,Asia,Bachelor's,N,N,98,1897,West,83434.0300,Year,Y,Denied
4,EZYV05,Africa,Master's,Y,N,1082,2005,South,149907.3900,Year,Y,Certified


In [35]:
usvisa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25480 entries, 0 to 25479
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   case_id                25480 non-null  object 
 1   continent              25480 non-null  object 
 2   education_of_employee  25480 non-null  object 
 3   has_job_experience     25480 non-null  object 
 4   requires_job_training  25480 non-null  object 
 5   no_of_employees        25480 non-null  int64  
 6   yr_of_estab            25480 non-null  int64  
 7   region_of_employment   25480 non-null  object 
 8   prevailing_wage        25480 non-null  float64
 9   unit_of_wage           25480 non-null  object 
 10  full_time_position     25480 non-null  object 
 11  case_status            25480 non-null  object 
dtypes: float64(1), int64(2), object(9)
memory usage: 2.3+ MB


# Data Drift Dashboard

In [36]:
usvisa.shape

(25480, 12)

In [41]:
df = usvisa.sample(frac=1, random_state=42).reset_index(drop=True)

ref = df.iloc[:5000]
cur = df.iloc[5000:10000]


In [42]:
drop_cols = ["case_id"]
ref = ref.drop(columns=drop_cols, errors="ignore")
cur = cur.drop(columns=drop_cols, errors="ignore")

In [43]:
def constant_cols(df):
    return [c for c in df.columns if df[c].nunique(dropna=False) <= 1]

print("Constant in ref:", constant_cols(ref))
print("Constant in cur:", constant_cols(cur))


Constant in ref: []
Constant in cur: []


In [48]:
usvisa_data_drift_dashboard = Dashboard(tabs=[DataDriftTab()])



In [53]:
mapping = ColumnMapping(
    numerical_features=["no_of_employees", "yr_of_estab", "prevailing_wage"],
    categorical_features=["continent","education_of_employee","has_job_experience",
                          "requires_job_training","region_of_employment","unit_of_wage","full_time_position"]
)

usvisa_data_drift_dashboard.calculate(ref, cur, column_mapping=mapping)

In [52]:
usvisa_data_drift_dashboard.show()

In [51]:
usvisa_data_drift_dashboard.save("usvisa_data_drift_report.html")

In [27]:
usvisa_data_drift_profile = Profile(sections=[DataDriftProfileSection()])

In [28]:
usvisa_data_drift_profile.calculate(usvisa[:200], usvisa[200:])

/opt/anaconda3/envs/visa/lib/python3.8/site-packages/scipy/stats/_stats_py.py:7407: RuntimeWarning:

divide by zero encountered in divide

/opt/anaconda3/envs/visa/lib/python3.8/site-packages/scipy/stats/_stats_py.py:7407: RuntimeWarning:

divide by zero encountered in divide

/opt/anaconda3/envs/visa/lib/python3.8/site-packages/scipy/stats/_stats_py.py:7407: RuntimeWarning:

divide by zero encountered in divide



In [50]:
usvisa_data_drift_profile.json()

'{"data_drift": {"name": "data_drift", "datetime": "2026-01-01 21:47:32.418871", "data": {"utility_columns": {"date": null, "id": null, "target": null, "prediction": null}, "num_feature_names": ["no_of_employees", "prevailing_wage", "yr_of_estab"], "cat_feature_names": ["case_id", "case_status", "continent", "education_of_employee", "full_time_position", "has_job_experience", "region_of_employment", "requires_job_training", "unit_of_wage"], "text_feature_names": [], "datetime_feature_names": [], "target_names": null, "options": {"confidence": null, "drift_share": 0.5, "nbinsx": 10, "xbins": null}, "metrics": {"n_features": 12, "n_drifted_features": 7, "share_drifted_features": 0.5833333333333334, "dataset_drift": true, "no_of_employees": {"current_small_hist": {"x": [-26.0, 60183.5, 120393.0, 180602.5, 240812.0, 301021.5, 361231.0, 421440.5, 481650.0, 541859.5, 602069.0], "y": [1.6277552398529144e-05, 2.214051968963643e-07, 5.978597304916662e-08, 1.2482785581694129e-08, 1.4453751726172